In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8 as required
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the working directory)
metric = pd.read_csv('metric.csv', low_memory=False)
trace = pd.read_csv('trace.csv', low_memory=False)
log = pd.read_csv('log.csv', low_memory=False)
error_logs = pd.read_csv('error_logs.csv', low_memory=False)

# Convert timestamp columns to timezone-aware datetimes (UTC -> Asia/Shanghai)
# Keep original numeric timestamp column; add a compact datetime column for inspection
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregates
metric_cmdb_counts = metric['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
metric_kpi_counts = metric['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count').head(50)
metric_total_rows = metric.shape[0]
metric_head = metric.head(5)

# 2) trace.csv aggregates
trace_cmdb_counts = trace['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
trace_name_counts = trace['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count').head(50)
trace_total_rows = trace.shape[0]
trace_head = trace.head(5)

# 3) log.csv aggregates
log_cmdb_counts = log['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
log_name_counts = log['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(50)
log_total_rows = log.shape[0]
log_head = log.head(5)

# 4) error_logs.csv aggregates
error_cmdb_counts = error_logs['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
error_total_rows = error_logs.shape[0]
error_head = error_logs.head(5)  # for message inspection (up to first 5 rows)

# Compact summaries (small outputs only)
metric_cmdb_counts, metric_kpi_counts, metric_total_rows, metric_head, \
trace_cmdb_counts, trace_name_counts, trace_total_rows, trace_head, \
log_cmdb_counts, log_name_counts, log_total_rows, log_head, \
error_cmdb_counts, error_total_rows, error_head

```
Out[1]:
```
# Summarize the previously computed aggregates in plain English as a single compact string.
def top_list(df, id_col):
    if df is None or df.shape[0] == 0:
        return "none"
    return ", ".join(f"{row[id_col]} ({int(row['count'])})" for _, row in df.head(5).iterrows())

summary = (
    "Telemetry file summary (compact):\n\n"
    f"1) metric.csv: total rows = {metric_total_rows}\n"
    f"   - Top cmdb_id (top 5): {top_list(metric_cmdb_counts, 'cmdb_id')}\n"
    f"   - Top kpi_name (top 5): {', '.join(metric_kpi_counts['kpi_name'].head(5).tolist())}\n"
    f"   - First 5-row preview available (columns: timestamp, cmdb_id, kpi_name, value, ts_dt).\n\n"
    f"2) trace.csv: total rows = {trace_total_rows}\n"
    f"   - Top cmdb_id (top 5): {top_list(trace_cmdb_counts, 'cmdb_id')}\n"
    f"   - Top trace_name (top 5): {', '.join(trace_name_counts['trace_name'].head(5).tolist())}\n"
    f"   - First 5-row preview available (columns: timestamp, cmdb_id, trace_name, value, ts_dt).\n\n"
    f"3) log.csv: total rows = {log_total_rows}\n"
    f"   - Top cmdb_id (top 5): {top_list(log_cmdb_counts, 'cmdb_id')}\n"
    f"   - Top log_name (top 5): {', '.join(log_name_counts['log_name'].head(5).tolist())}\n"
    f"   - First 5-row preview available (columns: timestamp, cmdb_id, log_name, value, ts_dt).\n\n"
    f"4) error_logs.csv: total rows = {error_total_rows}\n"
    f"   - Top cmdb_id (top 5): {top_list(error_cmdb_counts, 'cmdb_id')}\n"
    f"   - First 5-row preview: " + ("empty (no rows)" if error_head.shape[0]==0 else "shown in preview") + ".\n\n"
    "Notes:\n"
    "- metric.csv contains the largest dataset (27,829 rows). Top metric cmdbs include Mysql02, Mysql01, Redis01, Redis02, Tomcat04.\n"
    "- trace.csv has 6,975 rows; common trace names include trace.self.duration_mean/p95/row_count.\n"
    "- log.csv is smaller (524 rows); primary log_name entries are log.error_count and log.row_count.\n"
    "- error_logs.csv contains no rows in this dataset.\n"
    "- First-5-row previews for each file were loaded to confirm schema."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id  count
0        Mysql02   4010
1        Mysql01   3954
2        Redis01   2266
3        Redis02   2213
4       Tomcat04   1588
..           ...    ...
24  ServiceTest7    124
25      dockerB1     31
26      dockerA2     29
27      dockerB2     17
28      dockerA1     11

[29 rows x 2 columns],                                             kpi_name  count
0     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    368
1       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    367
2      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    360
3         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    360
4                 OSLinux-OSLinux_ZABBIX_Host_Uptime    360
..                                               ...    ...
45  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite    180
46   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps    180
47                   OSLinux-CPU_CPU-0_SingleCpuidle    180
48    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    179
49    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    179

[50 rows x 2 columns], 27829,     timestamp cmdb_id                                     kpi_name         value                     ts_dt
0  1615539600    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-12 17:00:00+08:00
1  1615539600    IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  1.837600e+01 2021-03-12 17:00:00+08:00
2  1615539600    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.453287e+08 2021-03-12 17:00:00+08:00
3  1615539600    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.789051e+07 2021-03-12 17:00:00+08:00
4  1615539600    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad  4.585000e-01 2021-03-12 17:00:00+08:00,      cmdb_id  count
0       MG02   1188
1       MG01   1182
2   Tomcat03    465
3   Tomcat01    465
4   dockerA2    465
5   dockerB1    462
6   dockerB2    462
7       IG01    459
8   Tomcat04    459
9   Tomcat02    459
10      IG02    456
11  dockerA1    453,                            trace_name  count
0            trace.self.duration_mean    360
1             trace.self.duration_p95    360
2                trace.self.row_count    360
3          trace.to_MG01.duration_p95    240
4             trace.to_MG01.row_count    240
..                                ...    ...
45   trace.from_dockerB1.duration_p95     60
46  trace.from_dockerB1.duration_mean     60
47      trace.from_dockerA2.row_count     60
48   trace.from_dockerA2.duration_p95     60
49  trace.from_dockerA2.duration_mean     60

[50 rows x 2 columns], 6975,     timestamp cmdb_id                       trace_name       value                     ts_dt
0  1615539600    IG01         trace.self.duration_mean     0.50652 2021-03-12 17:00:00+08:00
1  1615539600    IG01          trace.self.duration_p95     2.15455 2021-03-12 17:00:00+08:00
2  1615539600    IG01             trace.self.row_count  1758.00000 2021-03-12 17:00:00+08:00
3  1615539600    IG01  trace.to_Tomcat01.duration_mean     0.49584 2021-03-12 17:00:00+08:00
4  1615539600    IG01   trace.to_Tomcat01.duration_p95     2.14310 2021-03-12 17:00:00+08:00,     cmdb_id  count
0  Tomcat02     92
1  Tomcat04     84
2  Tomcat01     83
3  Tomcat03     83
4  apache01     62
5  apache02     62
6   Mysql01     58,            log_name  count
0   log.error_count    215
1     log.row_count    215
2  log.gc_pause_max     94, 524,     timestamp   cmdb_id          log_name   value                     ts_dt
0  1615539600   Mysql01   log.error_count    0.00 2021-03-12 17:00:00+08:00
1  1615539600   Mysql01     log.row_count   17.00 2021-03-12 17:00:00+08:00
2  1615539600  Tomcat01   log.error_count    0.00 2021-03-12 17:00:00+08:00
3  1615539600  Tomcat01  log.gc_pause_max    0.05 2021-03-12 17:00:00+08:00
4  1615539600  Tomcat01     log.row_count  410.00 2021-03-12 17:00:00+08:00, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message, ts_dt]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse previously loaded DataFrames: metric, trace, log, error_logs
# If not present in the kernel, load them (safe reload)
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv', low_memory=False)
    metric['ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv', low_memory=False)
    trace['ts_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    log
except NameError:
    log = pd.read_csv('log.csv', low_memory=False)
    log['ts_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv', low_memory=False)
    error_logs['ts_dt'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Components of interest
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Incident window (inclusive)
start_ts = 1615539600
end_ts = 1615541400

# Helper to compute aggregations for a DataFrame with given key columns and value column name
def compute_group_aggregates(df, key_cols, value_col):
    df = df.copy()
    # Ensure numeric
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    # Filter to components list (on first key column which is cmdb_id)
    df = df[df[key_cols[0]].isin(components)]
    # Global/full-series aggregates (per group)
    agg = df.groupby(key_cols).agg(
        count=(value_col, 'count'),
        p50=(value_col, lambda x: x.quantile(0.5)),
        p95=(value_col, lambda x: x.quantile(0.95)),
        mean=(value_col, 'mean'),
        std=(value_col, 'std')
    ).reset_index()
    # count in window
    mask = (df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)
    in_win = df[mask].groupby(key_cols)[value_col].count().reset_index(name='count_in_window')
    # Merge and fill missing window counts with 0
    agg = agg.merge(in_win, on=key_cols, how='left')
    agg['count_in_window'] = agg['count_in_window'].fillna(0).astype(int)
    # Sort by window count desc then overall count desc
    agg = agg.sort_values(['count_in_window', 'count'], ascending=[False, False]).reset_index(drop=True)
    return agg

# 1) metric.csv aggregations
metric_agg = compute_group_aggregates(metric, ['cmdb_id', 'kpi_name'], 'value')
# Keep compact: select columns and round numeric columns for readability
metric_agg_display = metric_agg[['cmdb_id','kpi_name','count','p50','p95','mean','std','count_in_window']].copy()
metric_agg_display[['p50','p95','mean','std']] = metric_agg_display[['p50','p95','mean','std']].round(6)
metric_agg_top50 = metric_agg_display.head(50)

# 2) trace.csv aggregations
trace_agg = compute_group_aggregates(trace, ['cmdb_id', 'trace_name'], 'value')
trace_agg_display = trace_agg[['cmdb_id','trace_name','count','p50','p95','mean','std','count_in_window']].copy()
trace_agg_display[['p50','p95','mean','std']] = trace_agg_display[['p50','p95','mean','std']].round(6)
trace_agg_top50 = trace_agg_display.head(50)

# 3) log.csv aggregations
log_agg = compute_group_aggregates(log, ['cmdb_id', 'log_name'], 'value')
log_agg_display = log_agg[['cmdb_id','log_name','count','p50','p95','mean','std','count_in_window']].copy()
log_agg_display[['p50','p95','mean','std']] = log_agg_display[['p50','p95','mean','std']].round(6)
log_agg_top50 = log_agg_display.head(50)

# 4) error_logs.csv summary for the window
error_total_rows = error_logs.shape[0]
error_mask = (error_logs['timestamp'] >= start_ts) & (error_logs['timestamp'] <= end_ts)
error_window_df = error_logs[error_mask].sort_values('timestamp')
error_window_count = error_window_df.shape[0]
error_window_head = error_window_df.head(5)  # up to first 5 rows within window

# Compact summary for error logs
error_summary = pd.DataFrame([{
    'total_rows': error_total_rows,
    'window_rows': int(error_window_count)
}])

# Display compact results (variables only)
metric_agg_top50, trace_agg_top50, log_agg_top50, error_summary, error_window_head

```
Out[2]:
```
# Create a compact plain-English summary string from the previously computed aggregates
# Reuse existing kernel variables: metric_agg_top50, trace_agg_top50, log_agg_top50, error_summary, error_window_head

def mk_top_list(df, cols, n=5):
    if df is None or df.shape[0] == 0:
        return "none"
    rows = []
    for r in df.head(n).itertuples(index=False):
        # cols is tuple of (group_col1, group_col2)
        rows.append(f"{getattr(r, cols[0])}/{getattr(r, cols[1])} (window:{int(getattr(r,'count_in_window'))}, total:{int(getattr(r,'count'))})")
    return "; ".join(rows)

metric_groups = metric_agg_top50.shape[0] if 'metric_agg_top50' in globals() else 0
trace_groups = trace_agg_top50.shape[0] if 'trace_agg_top50' in globals() else 0
log_groups = log_agg_top50.shape[0] if 'log_agg_top50' in globals() else 0

metric_top5 = mk_top_list(metric_agg_top50, ('cmdb_id','kpi_name'), 5)
trace_top5 = mk_top_list(trace_agg_top50, ('cmdb_id','trace_name'), 5)
log_top5 = mk_top_list(log_agg_top50, ('cmdb_id','log_name'), 5)

err_total = int(error_summary.at[0,'total_rows']) if ('error_summary' in globals() and error_summary.shape[0]>0) else 0
err_window = int(error_summary.at[0,'window_rows']) if ('error_summary' in globals() and error_summary.shape[0]>0) else 0
err_preview = "none"
if ('error_window_head' in globals() and error_window_head.shape[0]>0):
    # show up to first 5 rows timestamps and cmdb_id for inspection
    err_preview = "; ".join(f"{int(r.timestamp)}:{r.cmdb_id}" for r in error_window_head.itertuples())

summary = (
    "Aggregate summary (global stats per component-KPI, UTC+8):\n\n"
    f"1) metric.csv: groups returned = {metric_groups} (top 5 by window count):\n"
    f"   {metric_top5}\n\n"
    f"2) trace.csv: groups returned = {trace_groups} (top 5 by window count):\n"
    f"   {trace_top5}\n\n"
    f"3) log.csv: groups returned = {log_groups} (top 5 by window count):\n"
    f"   {log_top5}\n\n"
    f"4) error_logs.csv: total_rows = {err_total}, window_rows = {err_window}.\n"
    f"   Window preview (up to 5 rows): {err_preview}\n\n"
    "Notes: p50/p95/mean/std were computed over each group's entire series (global) and count_in_window\n"
    "gives the number of rows inside timestamp range [1615539600,1615541400].\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count           p50           p95          mean           std  count_in_window
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09  0.000000e+00               31
1      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  4.123413e+08  7.906410e+08  4.183409e+08  2.533836e+08               31
2      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.789051e+07  5.801434e+07  5.793407e+07  6.051673e+04               31
3      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.000000e+01  7.300000e+01  6.961290e+01  2.155264e+00               31
4      IG01                            OSLinux-CPU_CPU_CPULoad     31  1.020000e+00  1.220000e+00  1.052258e+00  7.684200e-02               31
..      ...                                                ...    ...           ...           ...           ...           ...              ...
45  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31
46  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31
47  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00               31
48  Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  6.000000e-01  6.000000e-01  5.580650e-01  5.016100e-02               31
49  Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  2.400000e+01  2.400000e+01  2.400000e+01  0.000000e+00               31

[50 rows x 8 columns],    cmdb_id                         trace_name  count          p50          p95         mean         std  count_in_window
0     IG01           trace.self.duration_mean     30     0.479035     0.598571     0.480111    0.095418               30
1     IG01            trace.self.duration_p95     30     2.090875     3.091550     2.158525    0.421378               30
2     IG01               trace.self.row_count     30  1665.000000  2055.300000  1652.066667  233.663910               30
3     IG01    trace.to_Tomcat01.duration_mean     30     0.446433     0.600409     0.468458    0.111222               30
4     IG01     trace.to_Tomcat01.duration_p95     30     2.078875     2.886110     2.026497    0.485804               30
..     ...                                ...    ...          ...          ...          ...         ...              ...
45    MG01  trace.from_dockerA2.duration_mean     30     0.054478     0.127036     0.061566    0.039099               30
46    MG01   trace.from_dockerA2.duration_p95     30     0.084275     1.017885     0.357557    0.414364               30
47    MG01      trace.from_dockerA2.row_count     30    39.500000    75.000000    43.333333   18.357247               30
48    MG01  trace.from_dockerB1.duration_mean     30     0.062354     0.100268     0.061486    0.022875               30
49    MG01   trace.from_dockerB1.duration_p95     30     0.446900     1.017997     0.491763    0.419170               30

[50 rows x 8 columns],      cmdb_id          log_name  count     p50        p95        mean         std  count_in_window
0   Tomcat01   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
1   Tomcat01     log.row_count     31  406.00   495.0000  395.258065   89.797909               31
2   Tomcat02   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
3   Tomcat02     log.row_count     31  407.00   531.0000  403.741935   90.750562               31
4   Tomcat03   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
5   Tomcat03     log.row_count     31  401.00   518.5000  401.903226   94.965030               31
6   Tomcat04   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
7   Tomcat04     log.row_count     31  412.00   490.5000  399.387097   88.529346               31
8   apache01   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
9   apache01     log.row_count     31  819.00  1010.0000  800.677419  178.845070               31
10  apache02   log.error_count     31    0.00     0.0000    0.000000    0.000000               31
11  apache02     log.row_count     31  798.00  1033.5000  796.483871  179.350657               31
12  Tomcat02  log.gc_pause_max     30    0.07     0.1055    0.071667    0.021186               30
13   Mysql01   log.error_count     29    0.00     0.0000    0.000000    0.000000               29
14   Mysql01     log.row_count     29    5.00    13.0000    5.862069    4.206493               29
15  Tomcat04  log.gc_pause_max     22    0.06     0.0995    0.061364    0.025127               22
16  Tomcat01  log.gc_pause_max     21    0.07     0.1100    0.065714    0.028031               21
17  Tomcat03  log.gc_pause_max     21    0.06     0.0800    0.059524    0.016576               21,    total_rows  window_rows
0           0            0, Empty DataFrame
Columns: [timestamp, cmdb_id, message, ts_dt]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from collections import defaultdict
import numpy as np

# Reuse existing metric DataFrame from kernel
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv', low_memory=False)
# Ensure timestamp dt in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in metric.columns:
    metric['ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure value numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Components of interest
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Incident window (inclusive)
start_ts = 1615539600
end_ts = 1615541400
window_mask = (metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)

# Filter metric to components of interest for grouping (global stats computed on full series per group)
metric_comp = metric[metric['cmdb_id'].isin(components)].copy()

# Compute global quantiles p5 and p95 and p50 (p95 used already earlier but recompute p95 to ensure p5 exists)
group_cols = ['cmdb_id', 'kpi_name']
# Use groupby.apply for quantiles to avoid issues with different group sizes
quantiles = metric_comp.groupby(group_cols)['value'].agg([
    ('count','count'),
    ('p5', lambda x: x.quantile(0.05)),
    ('p50', lambda x: x.quantile(0.50)),
    ('p95', lambda x: x.quantile(0.95)),
    ('mean','mean'),
    ('std','std')
]).reset_index()

# Count in window per group (must be computed after global quantiles)
metric_window = metric_comp[window_mask].copy()
count_in_window = metric_window.groupby(group_cols)['value'].count().reset_index(name='count_in_window')

# Merge quantiles with window counts (groups without window rows will be absent; we only care groups with count_in_window>0)
groups_with_window = quantiles.merge(count_in_window, on=group_cols, how='inner')

# Prepare containers for full per-group window-sliced series and anomalies list
window_series = dict()   # key: (cmdb_id,kpi_name) -> DataFrame of window rows for that group
anomaly_rows = []

# Iterate groups with window rows to compute exceed/drop counts and earliest timestamps and min/max in window
for _, row in groups_with_window.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['p95']
    p5 = row['p5']
    cnt_win = int(row['count_in_window'])
    # slice window series for this group
    df_win = metric_window[(metric_window['cmdb_id']==cmdb) & (metric_window['kpi_name']==kpi)].copy()
    # store full window-sliced series
    window_series[(cmdb, kpi)] = df_win  # saved in kernel for later inspection
    
    # compute exceed/drop counts and earliest timestamps
    # handle NaN thresholds gracefully
    if pd.isna(p95):
        exceed_count = 0
        exceed_earliest = pd.NaT
    else:
        exceed_mask = df_win['value'] >= p95
        exceed_count = int(exceed_mask.sum())
        if exceed_count > 0:
            exceed_earliest = df_win.loc[exceed_mask, 'ts_dt'].min()
        else:
            exceed_earliest = pd.NaT
    if pd.isna(p5):
        drop_count = 0
        drop_earliest = pd.NaT
    else:
        drop_mask = df_win['value'] <= p5
        drop_count = int(drop_mask.sum())
        if drop_count > 0:
            drop_earliest = df_win.loc[drop_mask, 'ts_dt'].min()
        else:
            drop_earliest = pd.NaT
    
    # max/min in window
    if df_win['value'].size > 0:
        max_in_win = df_win['value'].max()
        min_in_win = df_win['value'].min()
    else:
        max_in_win = np.nan
        min_in_win = np.nan
    
    anomaly_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'count_in_window': cnt_win,
        'p95': p95,
        'p5': p5,
        'exceed_count': exceed_count,
        'exceed_earliest_ts': exceed_earliest,
        'drop_count': drop_count,
        'drop_earliest_ts': drop_earliest,
        'max_value_in_window': max_in_win,
        'min_value_in_window': min_in_win
    })

# Create anomalies DataFrame
anomalies = pd.DataFrame(anomaly_rows)

# Replace NaT with None for cleaner display of earliest timestamps, but keep original types in variables
anomalies['exceed_earliest_ts'] = anomalies['exceed_earliest_ts'].where(anomalies['exceed_earliest_ts'].notna(), None)
anomalies['drop_earliest_ts'] = anomalies['drop_earliest_ts'].where(anomalies['drop_earliest_ts'].notna(), None)

# Compute sort key and sort descending by max(exceed_count, drop_count) then by count_in_window
anomalies['max_anom'] = anomalies[['exceed_count','drop_count']].max(axis=1)
anomalies = anomalies.sort_values(['max_anom','count_in_window'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20 groups for display
anomalies_top20 = anomalies.head(20).copy()

# Round numeric fields for compactness
anomalies_top20[['p95','p5','max_value_in_window','min_value_in_window']] = anomalies_top20[['p95','p5','max_value_in_window','min_value_in_window']].round(6)

# Drop helper column for final display
anomalies_top20 = anomalies_top20[['cmdb_id','kpi_name','count_in_window','p95','p5',
                                   'exceed_count','exceed_earliest_ts','drop_count','drop_earliest_ts',
                                   'max_value_in_window','min_value_in_window']]

# Save full anomalies and window_series in kernel variables for later inspection (per instruction)
# anomalies  -> full DataFrame for all groups with window rows
# window_series -> dict of full per-group window-sliced DataFrames
# Display compact top 20 anomalies
anomalies_top20, anomalies.shape, len(window_series)

```
Out[3]:
```
summary = (
    "Anomaly detection summary (incident window 2021-03-12 17:00:00+08:00 to 2021-03-12 17:30:00+08:00):\n\n"
    "- Groups evaluated with window data: 1627.\n"
    "- Top 20 groups (by max(exceed_count, drop_count)) were returned. Examples include:\n"
    "  * IG01 / JVM-Memory_7778_JVM_Memory_HeapMemoryMax (count_in_window=31, p95=p5=2.040136e+09, exceed_count=31, drop_count=31)\n"
    "  * IG01 / OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT (count_in_window=31, p95=p5=0, exceed_count=31, drop_count=31)\n"
    "  * IG02 / JVM-Memory_7778_JVM_Memory_HeapMemoryMax (count_in_window=31, p95=p5=2.040136e+09, exceed_count=31, drop_count=31)\n"
    "- Observation: many metric groups are constant (p5 == p95 and max == min in the window), so exceed_count and drop_count equal the full window count. That indicates values are equal to the global thresholds rather than clear up/down anomalies.\n"
    "- Implication: the current exceed/drop counts often reflect equality to percentile thresholds for constant series and should be inspected manually. Focus on groups where exceed_count>0 and drop_count==0 (or vice versa) for directional anomalies, but few such clear cases appear in the top results.\n"
    "- Saved kernel variables for further inspection: 'anomalies' (full per-group anomaly summary), 'anomalies_top20' (top 20 shown), and 'window_series' (dict of per-group window-sliced DataFrames). Use these to drill into raw values and timestamps for candidate groups.\n\n"
    "Recommended next steps: inspect the window_series for specific groups of interest to verify whether the flagged counts represent true deviations, and prioritize groups with non-equal exceed/drop behavior."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count_in_window           p95            p5  exceed_count        exceed_earliest_ts  drop_count          drop_earliest_ts  max_value_in_window  min_value_in_window
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31  2.040136e+09  2.040136e+09            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         2.040136e+09         2.040136e+09
1      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
2      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          30 2021-03-12 17:00:00+08:00         1.000000e+00         0.000000e+00
3      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31  1.000000e+00  1.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         1.000000e+00         1.000000e+00
4      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31  3.000000e-01  3.000000e-01            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         3.000000e-01         3.000000e-01
5      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31  3.000000e+00  3.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         3.000000e+00         3.000000e+00
6      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31  2.040136e+09  2.040136e+09            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         2.040136e+09         2.040136e+09
7      IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
8      IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
9      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31  1.000000e+00  1.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         1.000000e+00         1.000000e+00
10     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31  3.000000e+00  3.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         3.000000e+00         3.000000e+00
11     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31  2.040136e+09  2.040136e+09            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         2.040136e+09         2.040136e+09
12     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31  5.069495e+08  5.069495e+08            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         5.069495e+08         5.069495e+08
13     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31  2.040136e+09  2.040136e+09            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         2.040136e+09         2.040136e+09
14     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31  9.471521e+08  9.471521e+08            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         9.471521e+08         9.471521e+08
15  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
16  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
17  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
18  Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00
19  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31  0.000000e+00  0.000000e+00            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00         0.000000e+00         0.000000e+00, (1627, 12), 1627)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse anomalies DataFrame from kernel
try:
    anomalies
except NameError:
    raise NameError("anomalies DataFrame not found in kernel. Run anomaly detection step first.")

an = anomalies.copy()

# Ensure datetime columns are proper datetimes (coerce missing) and in Asia/Shanghai tz
tz = pytz.timezone('Asia/Shanghai')
an['exceed_earliest_ts'] = pd.to_datetime(an['exceed_earliest_ts'], errors='coerce')
an['drop_earliest_ts'] = pd.to_datetime(an['drop_earliest_ts'], errors='coerce')

# If timestamps are tz-naive, localize to Asia/Shanghai; if tz-aware, convert
def to_shanghai(series):
    if series.dropna().empty:
        return series
    sample = series.dropna().iloc[0]
    if sample.tzinfo is None:
        return series.dt.tz_localize(tz)
    else:
        return series.dt.tz_convert(tz)

an['exceed_earliest_ts'] = to_shanghai(an['exceed_earliest_ts'])
an['drop_earliest_ts'] = to_shanghai(an['drop_earliest_ts'])

# Group by component
group = an.groupby('cmdb_id')

# Build component_candidates DataFrame
component_candidates = pd.DataFrame({
    'cmdb_id': list(group.size().index)
}).set_index('cmdb_id')

component_candidates['total_groups_with_window'] = group['kpi_name'].nunique()
component_candidates['groups_with_exceed_only'] = group.apply(lambda g: ((g['exceed_count']>0) & (g['drop_count']==0)).sum())
component_candidates['groups_with_drop_only'] = group.apply(lambda g: ((g['drop_count']>0) & (g['exceed_count']==0)).sum())
component_candidates['groups_with_both'] = group.apply(lambda g: ((g['exceed_count']>0) & (g['drop_count']>0)).sum())

# earliest timestamps per component
component_candidates['earliest_exceed_ts'] = group['exceed_earliest_ts'].apply(lambda s: s.dropna().min() if not s.dropna().empty else pd.NaT)
component_candidates['earliest_drop_ts'] = group['drop_earliest_ts'].apply(lambda s: s.dropna().min() if not s.dropna().empty else pd.NaT)

# max fractions, handle divide-by-zero and NaNs
component_candidates['max_exceed_fraction'] = group.apply(
    lambda g: (g['exceed_count'] / g['count_in_window']).replace([np.inf, -np.inf], np.nan).fillna(0).max()
)
component_candidates['max_drop_fraction'] = group.apply(
    lambda g: (g['drop_count'] / g['count_in_window']).replace([np.inf, -np.inf], np.nan).fillna(0).max()
)

# Reset index and prepare sorting keys
component_candidates = component_candidates.reset_index()
component_candidates['tot_exceed_drop_only'] = component_candidates['groups_with_exceed_only'] + component_candidates['groups_with_drop_only']

# Sort as requested
component_candidates = component_candidates.sort_values(
    ['tot_exceed_drop_only', 'groups_with_both', 'total_groups_with_window'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Top 20 for display
component_candidates_top20 = component_candidates.head(20).copy()

# Round fraction columns for compactness
component_candidates_top20['max_exceed_fraction'] = component_candidates_top20['max_exceed_fraction'].round(3)
component_candidates_top20['max_drop_fraction'] = component_candidates_top20['max_drop_fraction'].round(3)

# Prepare top 3 components KPI lists (up to 5 each)
top3_components = component_candidates_top20['cmdb_id'].head(3).tolist()
top_components_kpis = {}

for comp in top3_components:
    comp_rows = an[(an['cmdb_id']==comp) & ((an['exceed_count']>0) | (an['drop_count']>0))].copy()
    if comp_rows.empty:
        top_components_kpis[comp] = pd.DataFrame(columns=['cmdb_id','kpi_name','count_in_window','exceed_count','exceed_earliest_ts',
                                                          'drop_count','drop_earliest_ts','max_value_in_window','min_value_in_window'])
        continue
    comp_rows['severity'] = comp_rows[['exceed_count','drop_count']].max(axis=1)
    comp_rows = comp_rows.sort_values(['severity','count_in_window'], ascending=[False, False])
    comp_display = comp_rows[['cmdb_id','kpi_name','count_in_window','exceed_count','exceed_earliest_ts',
                              'drop_count','drop_earliest_ts','max_value_in_window','min_value_in_window']].head(5).copy()
    comp_display[['max_value_in_window','min_value_in_window']] = comp_display[['max_value_in_window','min_value_in_window']].round(6)
    top_components_kpis[comp] = comp_display

# Save variables in kernel as requested
component_candidates = component_candidates  # full table
top_components_kpis = top_components_kpis

# Return compact outputs: top 20 component candidates and KPI lists for top 3 components
component_candidates_top20, top_components_kpis

```
Out[4]:
```
Summary of component-level candidate analysis (compact):

- Components evaluated: top 20 shown. Highest-ranked components (top rows) are:
  1. Mysql01 — total_groups_with_window = 224
  2. Mysql02 — total_groups_with_window = 224
  3. Redis01 — total_groups_with_window = 113
  4. Tomcat03 — 111
  5. Redis02 — 110
  6. Tomcat04 — 109
  7. Tomcat02 — 108
  8. Tomcat01 — 106
  9. MG02 — 94
  10. MG01 — 92
  11. IG01 — 88
  12. IG02 — 88
  13. apache01 — 80
  14. apache02 — 80
  (table limited to top 20; shown counts are total distinct KPI groups that had window data)

- Key aggregate findings (common across many components):
  - groups_with_exceed_only = 0 and groups_with_drop_only = 0 for these top components.
  - groups_with_both equals total_groups_with_window for these components (i.e., almost every group shows both exceed_count>0 and drop_count>0).
  - earliest_exceed_ts and earliest_drop_ts for these components are 2021-03-12 17:00:00+08:00 (the window start) in the top results.
  - max_exceed_fraction and max_drop_fraction are 1.0 for many components (meaning some groups had exceed_count/count_in_window == 1.0 and similarly for drop).

- Interpretation:
  - The anomaly flags are dominated by groups that simultaneously meet the exceed (>= p95) and drop (<= p5) conditions. This typically happens when the series is constant (p5 == p95) or when window values equal the global percentile thresholds — so exceed/drop counts simply reflect equality to the thresholds rather than directional deviations.
  - Because groups_with_both == total_groups_with_window and the exceed/drop earliest timestamps are the window start, these are likely not clear directional anomalies (increase-only or drop-only), but artifacts of constant/threshold-equal series.

- Top-3 components (detailed KPI candidates — up to 5 each):
  - Mysql01 (examples):
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — count_in_window=31, exceed_count=31, drop_count=31
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — count_in_window=31, exceed_count=31, drop_count=31
    - OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — count_in_window=30, exceed_count=30, drop_count=30
    - OSLinux-OSLinux_PROCESS_zabbix-... — count_in_window=30, exceed_count=30, drop_count=30
    - OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount — count_in_window=30, exceed_count=30, drop_count=30
  - Mysql02 (examples):
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — 31/31/31
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — 31/31/31
    - OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — 31/31/31
    - OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount — 31/31/31
    - Mysql-MySQL_3306_GetConnectedStateOfMysqld — 21/21/21
  - Redis01 (examples):
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT — 31/31/31
    - OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — 31/31/31
    - OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies — 31/31/31
    - OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount — 31/31/31
    - redis-Redis_6379_Redis_(aof_enabled) — 31/31/31

- Kernel artifacts saved for further inspection:
  - component_candidates (full per-component candidate table)
  - top_components_kpis (per-top-3 component KPI lists, each up to 5 rows)
  - window_series and anomalies (from the previous step) are still available to drill into raw window values and timestamps.

- Recommended next steps:
  1. Inspect window_series for representative groups where exceed/drop behavior is ambiguous to confirm whether values are constant or truly anomalous.
  2. Filter for directional candidates (groups_with_exceed_only > 0 or groups_with_drop_only > 0). Those are likelier to indicate real increases or decreases.
  3. For constant-series cases, consider excluding KPIs with p5 == p95 from directional anomaly scoring or use alternative thresholds (e.g., historical MAD) to avoid threshold-equality artifacts.

If you want, I can (a) list only groups with exceed_only or drop_only > 0, or (b) show raw window rows for a specific KPI from the top candidates for manual inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  total_groups_with_window  groups_with_exceed_only  groups_with_drop_only  groups_with_both        earliest_exceed_ts          earliest_drop_ts  max_exceed_fraction  max_drop_fraction  tot_exceed_drop_only
0    Mysql01                       224                        0                      0               224 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
1    Mysql02                       224                        0                      0               224 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
2    Redis01                       113                        0                      0               113 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
3   Tomcat03                       111                        0                      0               111 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
4    Redis02                       110                        0                      0               110 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
5   Tomcat04                       109                        0                      0               109 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
6   Tomcat02                       108                        0                      0               108 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
7   Tomcat01                       106                        0                      0               106 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
8       MG02                        94                        0                      0                94 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
9       MG01                        92                        0                      0                92 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
10      IG01                        88                        0                      0                88 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
11      IG02                        88                        0                      0                88 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
12  apache01                        80                        0                      0                80 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0
13  apache02                        80                        0                      0                80 2021-03-12 17:00:00+08:00 2021-03-12 17:00:00+08:00                  1.0                1.0                     0, {'Mysql01':      cmdb_id                                           kpi_name  count_in_window  exceed_count        exceed_earliest_ts  drop_count          drop_earliest_ts  max_value_in_window  min_value_in_window
15   Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
16   Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
142  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               30            30 2021-03-12 17:00:00+08:00          30 2021-03-12 17:00:00+08:00                  0.0                  0.0
143  Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               30            30 2021-03-12 17:00:00+08:00          30 2021-03-12 17:00:00+08:00                  0.4                  0.4
144  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               30            30 2021-03-12 17:00:00+08:00          30 2021-03-12 17:00:00+08:00                  7.0                  7.0, 'Mysql02':      cmdb_id                                        kpi_name  count_in_window  exceed_count        exceed_earliest_ts  drop_count          drop_earliest_ts  max_value_in_window  min_value_in_window
17   Mysql02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
18   Mysql02    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
19   Mysql02   OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
20   Mysql02      OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                 24.0                 24.0
152  Mysql02      Mysql-MySQL_3306_GetConnectedStateOfMysqld               21            21 2021-03-12 17:00:00+08:00          21 2021-03-12 17:00:00+08:00                  1.0                  1.0, 'Redis01':     cmdb_id                                        kpi_name  count_in_window  exceed_count        exceed_earliest_ts  drop_count          drop_earliest_ts  max_value_in_window  min_value_in_window
21  Redis01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
22  Redis01    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0
23  Redis01   OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  1.0                  1.0
24  Redis01      OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  4.0                  4.0
25  Redis01           redis-Redis_6379_Redis  (aof_enabled)               31            31 2021-03-12 17:00:00+08:00          31 2021-03-12 17:00:00+08:00                  0.0                  0.0})```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Check existence of 'anomalies' in kernel
if 'anomalies' not in globals():
    # Return single-line message as required
    "anomalies_not_found"
else:
    # Reuse anomalies DataFrame
    directional_anomalies = anomalies.copy()

    # Ensure datetime columns are proper datetimes and in Asia/Shanghai tz (UTC+8)
    tz = pytz.timezone('Asia/Shanghai')
    for col in ['exceed_earliest_ts', 'drop_earliest_ts']:
        if col in directional_anomalies.columns:
            directional_anomalies[col] = pd.to_datetime(directional_anomalies[col], errors='coerce')
            # If tz-naive, localize; if tz-aware, convert
            if not directional_anomalies[col].dropna().empty:
                sample = directional_anomalies[col].dropna().iloc[0]
                if sample.tzinfo is None:
                    directional_anomalies[col] = directional_anomalies[col].dt.tz_localize(tz)
                else:
                    directional_anomalies[col] = directional_anomalies[col].dt.tz_convert(tz)

    # 1) Filter directional anomalies:
    mask_spike = (directional_anomalies['exceed_count'] > 0) & (directional_anomalies['drop_count'] == 0)
    mask_drop  = (directional_anomalies['drop_count'] > 0) & (directional_anomalies['exceed_count'] == 0)
    mask_dir = mask_spike | mask_drop
    directional_anomalies = directional_anomalies[mask_dir].reset_index(drop=True)

    # Save variable in kernel (already assigned)
    # directional_anomalies

    # If empty, prepare empty outputs as specified
    if directional_anomalies.shape[0] == 0:
        overall_counts = pd.DataFrame([{
            'total_directional_groups': 0,
            'counts_by_type': {'spike': 0, 'drop': 0}
        }])
        per_component_summary = pd.DataFrame(columns=['cmdb_id','directional_group_count','spike_count','drop_count','earliest_directional_ts'])
        top_directional_groups = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','count_in_window','p95','p5','exceed_count','exceed_earliest_ts',
            'drop_count','drop_earliest_ts','exceed_fraction','drop_fraction','max_value_in_window','min_value_in_window'
        ])
    else:
        # 1) overall_counts
        n_spike = int((directional_anomalies['exceed_count'] > 0 & (directional_anomalies['drop_count'] == 0)).sum())
        # Note: parentheses needed around boolean expressions above; recompute correctly:
        n_spike = int(((directional_anomalies['exceed_count'] > 0) & (directional_anomalies['drop_count'] == 0)).sum())
        n_drop  = int(((directional_anomalies['drop_count'] > 0) & (directional_anomalies['exceed_count'] == 0)).sum())
        total_directional = int(directional_anomalies.shape[0])
        overall_counts = pd.DataFrame([{
            'total_directional_groups': total_directional,
            'counts_by_type': {'spike': n_spike, 'drop': n_drop}
        }])

        # 2) per_component_summary
        def earliest_directional_ts_for_group(df):
            # df contains only directional groups for a component
            # take min of available exceed/drop earliest timestamps
            ts = pd.to_datetime(pd.Series([], dtype='datetime64[ns]'))
            # collect non-null timestamps
            t1 = df['exceed_earliest_ts'].dropna() if 'exceed_earliest_ts' in df else pd.Series(dtype='datetime64[ns]')
            t2 = df['drop_earliest_ts'].dropna() if 'drop_earliest_ts' in df else pd.Series(dtype='datetime64[ns]')
            all_ts = pd.concat([t1, t2])
            if all_ts.empty:
                return pd.NaT
            # ensure tz-aware in Asia/Shanghai
            if all_ts.dt.tz is None:
                all_ts = all_ts.dt.tz_localize(tz)
            else:
                all_ts = all_ts.dt.tz_convert(tz)
            return all_ts.min()

        grp = directional_anomalies.groupby('cmdb_id')
        comp_rows = []
        for cmdb, g in grp:
            directional_group_count = int(g.shape[0])
            spike_count = int(((g['exceed_count'] > 0) & (g['drop_count'] == 0)).sum())
            drop_count = int(((g['drop_count'] > 0) & (g['exceed_count'] == 0)).sum())
            earliest_ts = earliest_directional_ts_for_group(g)
            comp_rows.append({
                'cmdb_id': cmdb,
                'directional_group_count': directional_group_count,
                'spike_count': spike_count,
                'drop_count': drop_count,
                'earliest_directional_ts': earliest_ts
            })
        per_component_summary = pd.DataFrame(comp_rows)
        per_component_summary = per_component_summary.sort_values('directional_group_count', ascending=False).reset_index(drop=True)
        per_component_summary['earliest_directional_ts'] = per_component_summary['earliest_directional_ts'].where(per_component_summary['earliest_directional_ts'].notna(), None)

        # Limit to top 20
        per_component_summary = per_component_summary.head(20)

        # 3) top_directional_groups
        df = directional_anomalies.copy()
        # compute fractions
        df['exceed_fraction'] = df['exceed_count'] / df['count_in_window']
        df['drop_fraction'] = df['drop_count'] / df['count_in_window']
        df['severity'] = df[['exceed_fraction','drop_fraction']].max(axis=1)
        # sort by severity desc then count_in_window desc
        df = df.sort_values(['severity','count_in_window'], ascending=[False, False]).reset_index(drop=True)

        # Select and format required columns
        cols = ['cmdb_id','kpi_name','count_in_window','p95','p5',
                'exceed_count','exceed_earliest_ts','drop_count','drop_earliest_ts',
                'exceed_fraction','drop_fraction','max_value_in_window','min_value_in_window']
        # Ensure datetime tz formatting preserved; replace NaT with None for display
        df['exceed_earliest_ts'] = df['exceed_earliest_ts'].where(df['exceed_earliest_ts'].notna(), None)
        df['drop_earliest_ts'] = df['drop_earliest_ts'].where(df['drop_earliest_ts'].notna(), None)

        top_directional_groups = df[cols].copy().head(20)
        # Round numeric columns for compactness
        for c in ['p95','p5','exceed_fraction','drop_fraction','max_value_in_window','min_value_in_window']:
            if c in top_directional_groups.columns:
                top_directional_groups[c] = top_directional_groups[c].round(6)

    # Final displayed variables (compact)
    overall_counts, per_component_summary, top_directional_groups

```
Out[7]:
```
Summary (plain English):

- Result: No directional-only anomalies were found. The filtered DataFrame 'directional_anomalies' is empty.

- overall_counts: total_directional_groups = 0 (spike = 0, drop = 0).

- per_component_summary: empty — no component has any directional-only groups in the window.

- top_directional_groups: empty — no directional groups to report.

- Context & recommendation: earlier analysis showed most metric groups in the window simultaneously met both exceed and drop conditions (exceed_count>0 and drop_count>0), typically because series were constant or window values equaled global percentile thresholds (p5 == p95). To find actionable anomalies, consider:
  1) excluding KPIs with p5 == p95 (constant series) from anomaly scoring, or
  2) using alternative anomaly rules (e.g., compare window median to historical median, use MAD or z-score, or require exceed_count/count_in_window above a threshold like 0.5) to surface directional spikes or drops.

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric DataFrame if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv', low_memory=False)

# Ensure timestamps and timezone (UTC+8) column present
tz = pytz.timezone('Asia/Shanghai')
if 'ts_dt' not in metric.columns:
    metric['ts_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    metric['ts_dt'] = pd.to_datetime(metric['ts_dt'], errors='coerce')
    # if tz-naive, localize; else convert
    if not metric['ts_dt'].dropna().empty:
        sample = metric['ts_dt'].dropna().iloc[0]
        if sample.tzinfo is None:
            metric['ts_dt'] = metric['ts_dt'].dt.tz_localize(tz)
        else:
            metric['ts_dt'] = metric['ts_dt'].dt.tz_convert(tz)

# Ensure numeric values
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Candidate components and incident window
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
start_ts = 1615539600
end_ts = 1615541400

# Work on subset of metric for these components
metric_comp = metric[metric['cmdb_id'].isin(components)].copy()

# Group-level global stats (median and MAD) over full series (not window)
grouped = metric_comp.groupby(['cmdb_id', 'kpi_name'])['value']
median_series = grouped.median().rename('median')
mad_series = grouped.apply(lambda x: np.median(np.abs(x - np.median(x)))).rename('mad')
count_series = grouped.count().rename('count')

stats = pd.concat([median_series, mad_series, count_series], axis=1).reset_index()

# Determine excluded groups (MAD == 0 or NaN)
stats['mad'] = pd.to_numeric(stats['mad'], errors='coerce')
stats['median'] = pd.to_numeric(stats['median'], errors='coerce')
excluded_mask = (stats['mad'] == 0) | (stats['mad'].isna())
excluded_groups = stats[excluded_mask].copy()
scored_groups = stats[~excluded_mask].copy()

# Prepare window slice
window_mask = (metric_comp['timestamp'] >= start_ts) & (metric_comp['timestamp'] <= end_ts)
metric_window = metric_comp[window_mask].copy()

# Containers to save per-point series annotated and group summaries
window_series_mad = dict()
mad_rows = []

# Helper to format earliest timestamp (keep tz-aware; convert NaT to None)
def earliest_ts_or_none(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    if pd.isna(s):
        return None
    return s  # should be tz-aware Timestamp

# Process scored groups (MAD > 0)
for _, row in scored_groups.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    median = float(row['median'])
    mad = float(row['mad'])
    # slice window points for this group
    df_win = metric_window[(metric_window['cmdb_id']==cmdb) & (metric_window['kpi_name']==kpi)].copy()
    # initialize per-point fields
    if not df_win.empty:
        # compute modified z per instruction
        df_win['modified_z'] = 0.6745 * (df_win['value'] - median) / mad
        df_win['abs_modified_z'] = df_win['modified_z'].abs()
        df_win['is_anomaly'] = df_win['abs_modified_z'] >= 3.5
        # earliest anomaly timestamp (UTC+8)
        if df_win['is_anomaly'].any():
            earliest_anom_ts = df_win.loc[df_win['is_anomaly'], 'ts_dt'].min()
            # determine direction(s)
            flagged = df_win.loc[df_win['is_anomaly']]
            directions = np.sign(flagged['value'] - median)
            # direction: 1 -> spike, -1 -> drop
            if (directions > 0).all():
                direction_label = 'spike'
            elif (directions < 0).all():
                direction_label = 'drop'
            else:
                direction_label = 'mixed'
            anomaly_count = int(flagged.shape[0])
            max_abs_z = float(df_win['abs_modified_z'].max())
        else:
            earliest_anom_ts = None
            direction_label = None
            anomaly_count = 0
            max_abs_z = float(df_win['abs_modified_z'].max()) if 'abs_modified_z' in df_win and not df_win['abs_modified_z'].isna().all() else 0.0
        max_val = float(df_win['value'].max())
        min_val = float(df_win['value'].min())
        count_in_window = int(df_win.shape[0])
    else:
        # no points in window
        df_win['modified_z'] = np.nan
        df_win['abs_modified_z'] = np.nan
        df_win['is_anomaly'] = False
        earliest_anom_ts = None
        direction_label = None
        anomaly_count = 0
        max_abs_z = np.nan
        max_val = np.nan
        min_val = np.nan
        count_in_window = 0

    # Save per-point annotated series keyed by tuple (cmdb,kpi)
    window_series_mad[(cmdb, kpi)] = df_win

    mad_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'count_in_window': count_in_window,
        'median': median,
        'mad': mad,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_ts': earliest_anom_ts,
        'anomaly_direction': direction_label,
        'max_abs_modified_z_in_window': (None if (isinstance(max_abs_z, float) and np.isnan(max_abs_z)) else max_abs_z),
        'max_value_in_window': max_val,
        'min_value_in_window': min_val
    })

# For excluded groups (MAD==0), record group summary but do not compute modified z (per instruction)
for _, row in excluded_groups.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    median = float(row['median']) if not pd.isna(row['median']) else np.nan
    mad = float(row['mad']) if not pd.isna(row['mad']) else np.nan
    df_win = metric_window[(metric_window['cmdb_id']==cmdb) & (metric_window['kpi_name']==kpi)].copy()
    # annotate but with NaNs
    if not df_win.empty:
        df_win['modified_z'] = np.nan
        df_win['abs_modified_z'] = np.nan
        df_win['is_anomaly'] = False
        count_in_window = int(df_win.shape[0])
        max_val = float(df_win['value'].max())
        min_val = float(df_win['value'].min())
    else:
        count_in_window = 0
        max_val = np.nan
        min_val = np.nan
    window_series_mad[(cmdb, kpi)] = df_win
    mad_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'count_in_window': count_in_window,
        'median': median,
        'mad': mad,
        'anomaly_count': 0,
        'earliest_anomaly_ts': None,
        'anomaly_direction': None,
        'max_abs_modified_z_in_window': None,
        'max_value_in_window': max_val,
        'min_value_in_window': min_val
    })

# Build group-level summary DataFrame
mad_anomalies = pd.DataFrame(mad_rows)

# Normalize earliest_anomaly_ts column: keep tz-aware timestamps or None
mad_anomalies['earliest_anomaly_ts'] = mad_anomalies['earliest_anomaly_ts'].where(mad_anomalies['earliest_anomaly_ts'].notna(), None)

# Compute totals
groups_evaluated = int(scored_groups.shape[0])          # groups with MAD>0 (evaluated)
num_excluded = int(excluded_groups.shape[0])           # groups excluded due to MAD==0
num_groups_with_anomalies = int((mad_anomalies['anomaly_count'] > 0).sum())

# Sort and prepare top 20 for display: by anomaly_count desc, then max_abs_modified_z_in_window desc
# Ensure max_abs_modified_z_in_window numeric for sorting (NaN -> -inf)
mad_anomalies['max_abs_z_sort'] = pd.to_numeric(mad_anomalies['max_abs_modified_z_in_window'], errors='coerce').fillna(-np.inf)
mad_top20 = mad_anomalies.sort_values(['anomaly_count', 'max_abs_z_sort'], ascending=[False, False]).head(20).copy()
mad_top20 = mad_top20.drop(columns=['max_abs_z_sort'])

# Round numeric columns for compact display
for c in ['median','mad','max_abs_modified_z_in_window','max_value_in_window','min_value_in_window']:
    if c in mad_top20.columns:
        mad_top20[c] = pd.to_numeric(mad_top20[c], errors='coerce').round(6)

# Save variables in kernel as requested
window_series_mad = window_series_mad   # dict of per-point window series annotated
mad_anomalies = mad_anomalies           # full group-level summary

# Final compact outputs
mad_top20, groups_evaluated, num_excluded, num_groups_with_anomalies

```
Out[8]:
```
Summary of MAD-based metric anomaly detection (incident window 2021-03-12 17:00:00+08:00 to 17:30:00+08:00):

- Scope:
  - Total metric groups for candidate components: 1627 (589 groups had MAD>0 and were evaluated; 1038 groups were excluded because MAD == 0 or NaN).
  - Groups with at least one MAD anomaly in the window: 190.

- Top findings (compact examples from top-ranked groups):
  1. IG01 / JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — count_in_window=31, median≈5.789e7, mad=1672, anomaly_count=11, earliest_anomaly=2021-03-12 17:20:00+08:00, direction=spike, max_abs_mod_z≈49.95.
  2. apache01 / OSLinux-CPU_CPU_CPUWio — count_in_window=31, median≈0.020759, mad≈0.012459, anomaly_count=10, earliest=2021-03-12 17:00:00+08:00, direction=spike, max_abs_mod_z≈21.81.
  3. Mysql02 / Mysql-MySQL_3306_Innodb buffer pool pages misc — count_in_window=20, median≈0.0333, mad≈0.00055, anomaly_count=9, earliest=2021-03-12 17:00:00+08:00, direction=mixed, max_abs_mod_z≈59.72.
  4. Redis02 / OSLinux-CPU_CPU_CPULoad — count_in_window=31, median=0.11, mad=0.07, anomaly_count=8, earliest=2021-03-12 17:14:00+08:00, direction=spike.
  5. Redis01 / OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn — count_in_window=16, median≈4283.5, mad≈1111.5, anomaly_count=7, earliest=2021-03-12 17:00:00+08:00, direction=spike.
  (Top-20 table saved in kernel as 'mad_top20'; full per-group summaries saved as 'mad_anomalies'.)

- Pattern notes:
  - Many anomalies are positive spikes (value > median) but there are also drops and mixed-direction groups.
  - Earliest detected anomalies often occur at or shortly after the window start (17:00), with some later spikes (e.g., 17:14, 17:20).
  - A large number of groups were excluded because MAD==0 (constant/near-constant series); these were recorded but not scored.

- Artifacts saved in kernel for follow-up:
  - window_series_mad: per-group window-sliced per-point series annotated with modified_z and anomaly flag (dictionary keyed by (cmdb_id, kpi_name)).
  - mad_anomalies: full per-group summary (one row per group).
  - mad_top20: compact top-20 groups table returned above.

- Recommended next steps:
  1. Inspect window_series_mad for the top groups (by anomaly_count and max_abs_modified_z) to confirm real incidents vs noisy/metric-artifact spikes.
  2. Prioritize groups with high anomaly_count and large max_abs_modified_z for investigation (e.g., the examples above).
  3. For excluded groups (MAD==0), consider alternate methods (e.g., delta from baseline or monitoring metadata) if those KPIs matter operationally.

If you want, I can list the top-20 table rows explicitly or extract the per-point series for a specific (cmdb_id, kpi_name) from window_series_mad for inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                        kpi_name  count_in_window        median          mad  anomaly_count       earliest_anomaly_ts anomaly_direction  max_abs_modified_z_in_window  max_value_in_window  min_value_in_window
2        IG01     JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed               31  5.789051e+07  1672.000000             11 2021-03-12 17:20:00+08:00             spike                  4.995495e+01         5.801434e+07         5.788884e+07
533  apache01                          OSLinux-CPU_CPU_CPUWio               31  2.075900e-02     0.012459             10 2021-03-12 17:00:00+08:00             spike                  2.180883e+01         4.236000e-01         8.300000e-03
231   Mysql02  Mysql-MySQL_3306_Innodb buffer pool pages misc               20  3.330000e-02     0.000550              9 2021-03-12 17:00:00+08:00             mixed                  5.972391e+01         8.200000e-02         1.670000e-02
338   Redis02                         OSLinux-CPU_CPU_CPULoad               31  1.100000e-01     0.070000              8 2021-03-12 17:14:00+08:00             spike                  3.757929e+01         4.010000e+00         0.000000e+00
318   Redis01     OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn               16  4.283500e+03  1111.500000              7 2021-03-12 17:00:00+08:00             spike                  3.783992e+06         6.235597e+09         2.872000e+03
35       IG01     OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn               16  2.545000e+02    79.500000              7 2021-03-12 17:02:00+08:00             spike                  3.664588e+06         4.319273e+08         1.700000e+02
36       IG01    OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut               16  2.395000e+02    75.500000              7 2021-03-12 17:02:00+08:00             spike                  3.589987e+06         4.018446e+08         1.590000e+02
319   Redis01    OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut               16  4.090000e+03  1101.500000              7 2021-03-12 17:00:00+08:00             spike                  3.475245e+06         5.675293e+09         2.694000e+03
341   Redis02                          OSLinux-CPU_CPU_CPUWio               31  8.400000e-03     0.004200              7 2021-03-12 17:00:00+08:00             spike                  8.117961e+01         5.138920e-01         0.000000e+00
46       IG02                         OSLinux-CPU_CPU_CPULoad               31  1.010000e+00     0.010000              7 2021-03-12 17:02:00+08:00             spike                  1.753700e+01         1.270000e+00         1.000000e+00
564  apache02                         OSLinux-CPU_CPU_CPULoad               31  1.000000e-02     0.010000              7 2021-03-12 17:01:00+08:00             spike                  1.079200e+01         1.700000e-01         0.000000e+00
340   Redis02                     OSLinux-CPU_CPU_CPUUserTime               31  3.306000e-01     0.037062              6 2021-03-12 17:14:00+08:00             spike                  1.793041e+03         9.885350e+01         2.423430e-01
342   Redis02                     OSLinux-CPU_CPU_CPUidleutil               31  9.856557e+01     0.155726              6 2021-03-12 17:14:00+08:00              drop                  4.268296e+02         9.889400e+01         2.077000e-02
305   Redis01                          OSLinux-CPU_CPU_CPUWio               31  1.250000e-02     0.004200              6 2021-03-12 17:00:00+08:00             spike                  3.748293e+01         2.459000e-01         4.174000e-03
14       IG01                          OSLinux-CPU_CPU_CPUWio               31  5.820000e-02     0.041568              6 2021-03-12 17:00:00+08:00             spike                  1.093520e+01         7.321130e-01         1.660000e-02
524  apache01                 OSLinux-CPU_CPU-1_SingleCpuidle               15  9.127116e+01     8.263957              6 2021-03-12 17:05:00+08:00              drop                  7.449506e+00         9.953512e+01         0.000000e+00
73       IG02    OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut               15  2.100000e+02    36.000000              5 2021-03-12 17:01:00+08:00             spike                  7.554128e+06         4.031857e+08         1.640000e+02
72       IG02     OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn               15  2.240000e+02    39.000000              5 2021-03-12 17:01:00+08:00             spike                  7.494840e+06         4.333564e+08         1.750000e+02
442  Tomcat03                 OSLinux-CPU_CPU-1_SingleCpuUtil               11  3.896800e+00     1.234100              5 2021-03-12 17:01:00+08:00             spike                  4.330262e+01         8.312550e+01         2.662700e+00
90       MG01                          OSLinux-CPU_CPU_CPUWio               18  2.910000e-02     0.014550              5 2021-03-12 17:01:00+08:00             spike                  3.450380e+01         7.734000e-01         1.250000e-02, 589, 1038, 190)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Check mad_anomalies presence
if 'mad_anomalies' not in globals():
    "mad_anomalies_not_found"
else:
    # Use existing variables from kernel: mad_anomalies, trace, log, error_logs
    tz = pytz.timezone('Asia/Shanghai')

    # Ensure trace, log, error_logs loaded and have ts_dt in UTC+8
    for df_name in ('trace','log','error_logs'):
        if df_name not in globals():
            raise NameError(f"{df_name} not found in kernel")
        df = globals()[df_name]
        if 'ts_dt' not in df.columns:
            df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df['ts_dt'] = pd.to_datetime(df['ts_dt'], errors='coerce')
            if not df['ts_dt'].dropna().empty:
                sample = df['ts_dt'].dropna().iloc[0]
                if sample.tzinfo is None:
                    df['ts_dt'] = df['ts_dt'].dt.tz_localize(tz)
                else:
                    df['ts_dt'] = df['ts_dt'].dt.tz_convert(tz)
        globals()[df_name] = df

    # 1) Top components by metric anomalies
    ma = mad_anomalies.copy()
    ma_dir = ma[ma['anomaly_count'] > 0].copy()
    if ma_dir.empty:
        top_components_df = pd.DataFrame(columns=['cmdb_id','groups_with_anomalies','total_anomalies','earliest_anomaly_ts'])
        trace_candidates = {'trace_agg': pd.DataFrame(), 'trace_edges': pd.DataFrame()}
        log_candidates = {'log_agg': pd.DataFrame(), 'error_excerpt': pd.DataFrame()}
        top_components_df, trace_candidates, log_candidates
    else:
        comp_grp = ma_dir.groupby('cmdb_id')
        comp_summary = comp_grp.agg(
            groups_with_anomalies=('kpi_name','nunique'),
            total_anomalies=('anomaly_count','sum'),
            earliest_anomaly_ts=('earliest_anomaly_ts', lambda s: pd.to_datetime(s.dropna()).min() if not s.dropna().empty else pd.NaT)
        ).reset_index()
        comp_summary['earliest_anomaly_ts'] = comp_summary['earliest_anomaly_ts'].where(comp_summary['earliest_anomaly_ts'].notna(), None)
        top_components_df = comp_summary.sort_values(['groups_with_anomalies','total_anomalies'], ascending=[False, False]).head(5).reset_index(drop=True)

        # Top 5 component list
        top5 = top_components_df['cmdb_id'].tolist()

        # Incident window mask based on integer timestamp range
        start_ts = 1615539600
        end_ts = 1615541400

        # 2) Trace inspection per top component (window-restricted)
        trace_df = trace.copy()
        trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts) & (trace_df['cmdb_id'].isin(top5))].copy()

        # a) aggregated trace_name stats per component (window only)
        trace_agg_rows = []
        for comp in top5:
            comp_tr = trace_window[trace_window['cmdb_id']==comp]
            if comp_tr.empty:
                continue
            grp = comp_tr.groupby('trace_name')['value']
            agg = grp.agg(count_in_window='count', p50=lambda x: x.quantile(0.5), p95=lambda x: x.quantile(0.95), mean='mean').reset_index()
            agg['cmdb_id'] = comp
            # reorder and round
            agg = agg[['cmdb_id','trace_name','count_in_window','p50','p95','mean']]
            agg[['p50','p95','mean']] = agg[['p50','p95','mean']].round(6)
            agg = agg.sort_values('count_in_window', ascending=False).head(10)
            trace_agg_rows.append(agg)
        if trace_agg_rows:
            trace_agg_all = pd.concat(trace_agg_rows, ignore_index=True)
        else:
            trace_agg_all = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','p50','p95','mean'])

        # b) extract trace edges trace.to_{peer}.* and trace.from_{peer}.*
        edge_pattern = re.compile(r'^trace\.(to|from)_(?P<peer>[^.]+)\.(?P<suffix>.+)$')
        edges = []
        for comp in top5:
            comp_tr = trace_window[trace_window['cmdb_id']==comp]
            if comp_tr.empty:
                continue
            comp_tr = comp_tr.copy()
            # parse trace_name
            parsed = comp_tr['trace_name'].str.extract(r'^trace\.(to|from)_(?P<peer>[^.]+)\.(?P<suffix>.+)$', expand=True)
            # parsed has columns [0, 'peer', 'suffix'] with 0 being 'to'/'from'
            comp_tr[['edge_type','peer_component','metric_type_suffix']] = parsed[[0,'peer','suffix']]
            edge_rows = comp_tr[comp_tr['edge_type'].notna()].copy()
            if edge_rows.empty:
                continue
            grp = edge_rows.groupby(['cmdb_id','edge_type','peer_component','metric_type_suffix'])['value']
            agg = grp.agg(count_in_window='count', p95=lambda x: x.quantile(0.95), mean='mean').reset_index()
            agg['p95'] = agg['p95'].round(6)
            agg['mean'] = agg['mean'].round(6)
            # mark downstream_of: if edge_type == 'to' and peer_component is in top5, it's downstream_of peer
            agg['downstream_of'] = agg.apply(lambda r: r['peer_component'] if (r['edge_type']=='to' and r['peer_component'] in top5) else None, axis=1)
            edges.append(agg)
        if edges:
            trace_edges_df = pd.concat(edges, ignore_index=True)
            trace_edges_top20 = trace_edges_df.sort_values('count_in_window', ascending=False).head(20).reset_index(drop=True)
        else:
            trace_edges_df = pd.DataFrame(columns=['cmdb_id','edge_type','peer_component','metric_type_suffix','count_in_window','p95','mean','downstream_of'])
            trace_edges_top20 = trace_edges_df.copy()

        # Save trace_candidates
        trace_candidates = {'trace_agg': trace_agg_all, 'trace_edges': trace_edges_df}

        # 3) Log inspection per top component within window
        log_df = log.copy()
        log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts) & (log_df['cmdb_id'].isin(top5))].copy()

        log_agg_rows = []
        for comp in top5:
            comp_logs = log_window[log_window['cmdb_id']==comp]
            if comp_logs.empty:
                continue
            grp = comp_logs.groupby('log_name')['value']
            agg = grp.agg(count_in_window='count', p50=lambda x: x.quantile(0.5), p95=lambda x: x.quantile(0.95), mean='mean', max_value_in_window='max').reset_index()
            agg['cmdb_id'] = comp
            agg = agg[['cmdb_id','log_name','count_in_window','p50','p95','mean','max_value_in_window']]
            agg[['p50','p95','mean','max_value_in_window']] = agg[['p50','p95','mean','max_value_in_window']].round(6)
            agg = agg.sort_values('count_in_window', ascending=False).head(10)
            log_agg_rows.append(agg)
        if log_agg_rows:
            log_agg_all = pd.concat(log_agg_rows, ignore_index=True)
        else:
            log_agg_all = pd.DataFrame(columns=['cmdb_id','log_name','count_in_window','p50','p95','mean','max_value_in_window'])

        # Check error_logs for these top components in the window
        elogs = error_logs.copy()
        elogs_window = elogs[(elogs['timestamp'] >= start_ts) & (elogs['timestamp'] <= end_ts) & (elogs['cmdb_id'].isin(top5))].copy()
        if not elogs_window.empty:
            # keep first 5 rows per all components (not per component)
            elogs_window = elogs_window.sort_values('timestamp').head(5)[['timestamp','cmdb_id','message','ts_dt']].reset_index(drop=True)
        else:
            elogs_window = pd.DataFrame(columns=['timestamp','cmdb_id','message','ts_dt'])

        # Save log_candidates
        log_candidates = {'log_agg': log_agg_all, 'error_excerpt': elogs_window}

        # Final compact outputs (top components summary, top trace edges, aggregated traces, logs, error excerpt)
        # Keep outputs bounded: top_components_df (5 rows), trace_agg_all (top 10 per comp already bounded), trace_edges_top20 (<=20), log_agg_all (bounded), elogs_window (<=5)
        top_components_df, trace_agg_all, trace_edges_top20, log_agg_all, elogs_window

```
Out[9]:
```
The analysis ran successfully but the cell returned no printed tables. Results were saved in kernel variables for follow-up inspection:

- top_components_df — top 5 components by metric anomalies (groups_with_anomalies, total_anomalies, earliest_anomaly_ts).
- trace_candidates — dict with:
  - trace_candidates['trace_agg']: aggregated trace stats per top component (columns: cmdb_id, trace_name, count_in_window, p50, p95, mean).
  - trace_candidates['trace_edges']: extracted trace edges (to/from peer, metric suffix) with counts, p95, mean and a downstream_of marker when the peer is one of the top components.
- log_candidates — dict with:
  - log_candidates['log_agg']: aggregated log stats per top component (cmdb_id, log_name, count_in_window, p50, p95, mean, max_value_in_window).
  - log_candidates['error_excerpt']: up to 5 error_log rows (timestamp, cmdb_id, message, ts_dt) in the window for the top components (may be empty).

If you want to see the compact tables now, I can display them. Example quick checks I can run for you:
- top_components_df
- trace_candidates['trace_agg'].head(20)
- trace_candidates['trace_edges'].sort_values('count_in_window', ascending=False).head(20)
- log_candidates['log_agg'].head(20)
- log_candidates['error_excerpt']

Which one should I show?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing kernel variables; create safe fallbacks if missing
tz = pytz.timezone('Asia/Shanghai')

# 1) top_components_df (ensure expected columns)
if 'top_components_df' in globals() and isinstance(top_components_df, pd.DataFrame):
    top_components_display = top_components_df.loc[:, ['cmdb_id','groups_with_anomalies','total_anomalies','earliest_anomaly_ts']].head(5)
else:
    top_components_display = pd.DataFrame(columns=['cmdb_id','groups_with_anomalies','total_anomalies','earliest_anomaly_ts'])

# Determine top5 components list from the displayed summary
top5_components = top_components_display['cmdb_id'].dropna().tolist()

# 2) trace_candidates['trace_edges'] sorted by count_in_window desc, top 20
if 'trace_candidates' in globals() and isinstance(trace_candidates, dict) and 'trace_edges' in trace_candidates:
    trace_edges_df = trace_candidates['trace_edges']
else:
    # fallback if variable named trace_edges_df exists
    trace_edges_df = globals().get('trace_edges_df', pd.DataFrame())

# Ensure expected columns exist; if not, create empty frame with expected cols
edge_cols = ['cmdb_id','edge_type','peer_component','metric_type_suffix','count_in_window','p95','mean','downstream_of']
if trace_edges_df.empty:
    trace_edges_top20 = pd.DataFrame(columns=edge_cols)
else:
    # select and sort
    trace_edges_top20 = trace_edges_df.loc[:, [c for c in edge_cols if c in trace_edges_df.columns]].sort_values('count_in_window', ascending=False).head(20).reset_index(drop=True)
    # Ensure all expected columns present (fill missing with None)
    for c in edge_cols:
        if c not in trace_edges_top20.columns:
            trace_edges_top20[c] = None
    trace_edges_top20 = trace_edges_top20[edge_cols]

# 3) trace_candidates['trace_agg'] filtered to top5 components (up to 20 rows)
if 'trace_candidates' in globals() and isinstance(trace_candidates, dict) and 'trace_agg' in trace_candidates:
    trace_agg_all = trace_candidates['trace_agg']
else:
    trace_agg_all = globals().get('trace_agg_all', pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','p50','p95','mean']))

if not trace_agg_all.empty and top5_components:
    trace_agg_filtered = trace_agg_all[trace_agg_all['cmdb_id'].isin(top5_components)].copy().reset_index(drop=True)
    trace_agg_filtered = trace_agg_filtered[['cmdb_id','trace_name','count_in_window','p50','p95','mean']].head(20)
else:
    trace_agg_filtered = pd.DataFrame(columns=['cmdb_id','trace_name','count_in_window','p50','p95','mean'])

# 4) log_candidates['log_agg'] filtered to top5 components (up to 20 rows)
if 'log_candidates' in globals() and isinstance(log_candidates, dict) and 'log_agg' in log_candidates:
    log_agg_all = log_candidates['log_agg']
else:
    log_agg_all = globals().get('log_agg_all', pd.DataFrame(columns=['cmdb_id','log_name','count_in_window','p50','p95','mean','max_value_in_window']))

if not log_agg_all.empty and top5_components:
    log_agg_filtered = log_agg_all[log_agg_all['cmdb_id'].isin(top5_components)].copy().reset_index(drop=True)
    log_agg_filtered = log_agg_filtered[['cmdb_id','log_name','count_in_window','p50','p95','mean','max_value_in_window']].head(20)
else:
    log_agg_filtered = pd.DataFrame(columns=['cmdb_id','log_name','count_in_window','p50','p95','mean','max_value_in_window'])

# 5) error log excerpt for top5 components (up to 5 rows)
if 'log_candidates' in globals() and isinstance(log_candidates, dict) and 'error_excerpt' in log_candidates:
    error_excerpt = log_candidates['error_excerpt']
else:
    error_excerpt = globals().get('elogs_window', pd.DataFrame())

# Ensure we only show up to 5 rows and relevant columns if present
if not error_excerpt.empty:
    cols_keep = [c for c in ['timestamp','cmdb_id','message','ts_dt'] if c in error_excerpt.columns]
    error_excerpt = error_excerpt.loc[:, cols_keep].head(5).reset_index(drop=True)
else:
    error_excerpt = pd.DataFrame(columns=['timestamp','cmdb_id','message','ts_dt'])

# Display compact results as DataFrames (IPython will show them)
top_components_display, trace_edges_top20, trace_agg_filtered, log_agg_filtered, error_excerpt

```
Out[10]:
```
Compact summary of triage outputs (UTC+8):

1) Top 5 components by metric anomalies
- Mysql02 — groups_with_anomalies = 38, total_anomalies = 85, earliest_anomaly_ts = 2021-03-12 17:00:00+08:00
- Mysql01 — groups_with_anomalies = 28, total_anomalies = 51, earliest_anomaly_ts = 2021-03-12 17:00:00+08:00
- Tomcat02 — groups_with_anomalies = 15, total_anomalies = 21, earliest_anomaly_ts = 2021-03-12 17:00:00+08:00
- Tomcat04 — groups_with_anomalies = 14, total_anomalies = 22, earliest_anomaly_ts = 2021-03-12 17:03:00+08:00
- apache01 — groups_with_anomalies = 13, total_anomalies = 40, earliest_anomaly_ts = 2021-03-12 17:00:00+08:00

2) Trace edges (top 20 by count_in_window)
- Tomcat02: receives traces from IG01 and IG02 (trace.from_IG01.*, trace.from_IG02.*) and calls MG01/MG02 (trace.to_MG01.*, trace.to_MG02.*). Many edge metrics have count_in_window = 30 and non-trivial p95/mean values.
- Tomcat04: similar pattern — receives from IG01/IG02 and calls MG01/MG02 (count_in_window = 30).
- No edge rows in the top-20 had downstream_of set (downstream_of is None), meaning the peer component in those edges was not marked as one of the top-5 components in the current extraction (or edges were 'from' rather than 'to' in the relevant direction).
- Example rows: Tomcat02 <- from IG01 (duration_mean/p95/row_count), Tomcat02 -> MG01/MG02 (duration_mean/p95/row_count); Tomcat04 similar.

3) Trace aggregates for the top 5 components (examples, up to 20 rows)
- Tomcat02 top traces (count_in_window = 30): trace.from_IG01.duration_mean/p95/row_count; trace.from_IG02.duration_mean/p95/row_count; trace.self.duration_mean/row_count/duration_p95; trace.to_MG01.duration_mean.
- Tomcat04 similar set of traces (from IG01/IG02; to MG01; self.*). Values include p50/p95/mean summarized per trace_name over the window.

4) Log aggregates for the top 5 components (examples)
- Mysql01: log.error_count (count_in_window=29, p50=0), log.row_count (count_in_window=29, p50=5, max=17)
- Tomcat02: log.error_count (31, p50=0), log.row_count (31, p50=407, max=549), log.gc_pause_max (30)
- Tomcat04 and apache01: similar log.row_count and log.error_count entries; apache01 log.row_count is large (count_in_window=31, p50=819, max=1069).
- Error log excerpt: none found in the window for these top components (no error_logs rows in-window).

Saved kernel artifacts
- trace_candidates (trace_agg and trace_edges) and log_candidates (log_agg and error_excerpt) are saved for follow-up.

Interpretation & recommended next steps
- Metric anomalies point to Mysql01/Mysql02 and several Tomcat/apache nodes. Many metric anomalies start at the window start (17:00).  
- Tracing shows Tomcat02 and Tomcat04 are receiving traffic from IG01/IG02 and calling MG01/MG02 — these service-call edges are candidate paths to investigate for propagation of metric anomalies.  
- Logs show elevated row_count on web/app components (apache, Tomcat) at the same window; no error_log messages found in the window for the top components.  
- Next actions: inspect per-group window_series_mad for the highest-ranked metric groups, inspect trace edge timings to see whether issues propagate IG -> Tomcat -> MG / Mysql, and drill into Tomcat/Mysql metrics and logs for root-cause signals.

If you want, I can now display any one of these saved tables in full (bounded output) or extract per-point series for a specific (cmdb_id, kpi_name) from window_series_mad for manual inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  groups_with_anomalies  total_anomalies       earliest_anomaly_ts
0   Mysql02                     38               85 2021-03-12 17:00:00+08:00
1   Mysql01                     28               51 2021-03-12 17:00:00+08:00
2  Tomcat02                     15               21 2021-03-12 17:00:00+08:00
3  Tomcat04                     14               22 2021-03-12 17:03:00+08:00
4  apache01                     13               40 2021-03-12 17:00:00+08:00,      cmdb_id edge_type peer_component metric_type_suffix  count_in_window         p95        mean downstream_of
0   Tomcat02      from           IG01      duration_mean               30    0.651777    0.482870          None
1   Tomcat02      from           IG01       duration_p95               30    3.114808    2.174395          None
2   Tomcat02      from           IG01          row_count               30  277.400000  212.500000          None
3   Tomcat02      from           IG02      duration_mean               30    0.668558    0.484242          None
4   Tomcat02      from           IG02       duration_p95               30    2.416942    2.042335          None
5   Tomcat02      from           IG02          row_count               30  249.450000  202.966667          None
6   Tomcat02        to           MG01      duration_mean               30    0.529836    0.379431          None
7   Tomcat02        to           MG02      duration_mean               30    0.537193    0.363043          None
8   Tomcat02        to           MG01          row_count               30  255.050000  205.466667          None
9   Tomcat02        to           MG01       duration_p95               30    2.654477    1.998312          None
10  Tomcat02        to           MG02       duration_p95               30    3.091405    1.913650          None
11  Tomcat04      from           IG01       duration_p95               30    2.845655    2.135870          None
12  Tomcat04      from           IG01      duration_mean               30    0.620715    0.483817          None
13  Tomcat02        to           MG02          row_count               30  265.950000  209.900000          None
14  Tomcat04        to           MG02       duration_p95               30    3.109212    1.941878          None
15  Tomcat04        to           MG01      duration_mean               30    0.489026    0.376030          None
16  Tomcat04        to           MG02      duration_mean               30    0.576291    0.376993          None
17  Tomcat04      from           IG01          row_count               30  260.600000  205.866667          None
18  Tomcat04      from           IG02      duration_mean               30    0.652542    0.482431          None
19  Tomcat04      from           IG02       duration_p95               30    3.107395    2.141590          None,      cmdb_id                     trace_name  count_in_window          p50          p95         mean
0   Tomcat02  trace.from_IG01.duration_mean               30     0.440608     0.651777     0.482870
1   Tomcat02   trace.from_IG01.duration_p95               30     2.091125     3.114808     2.174395
2   Tomcat02      trace.from_IG01.row_count               30   213.000000   277.400000   212.500000
3   Tomcat02  trace.from_IG02.duration_mean               30     0.478518     0.668558     0.484242
4   Tomcat02   trace.from_IG02.duration_p95               30     2.106150     2.416942     2.042335
5   Tomcat02      trace.from_IG02.row_count               30   199.000000   249.450000   202.966667
6   Tomcat02       trace.self.duration_mean               30     0.066530     0.092372     0.069650
7   Tomcat02    trace.to_MG01.duration_mean               30     0.367281     0.529836     0.379431
8   Tomcat02           trace.self.row_count               30  2768.000000  3671.650000  2840.366667
9   Tomcat02        trace.self.duration_p95               30     0.117100     1.005000     0.400593
10  Tomcat04  trace.from_IG01.duration_mean               30     0.494830     0.620715     0.483817
11  Tomcat04   trace.from_IG01.duration_p95               30     2.107525     2.845655     2.135870
12  Tomcat04      trace.from_IG01.row_count               30   207.000000   260.600000   205.866667
13  Tomcat04  trace.from_IG02.duration_mean               30     0.475793     0.652542     0.482431
14  Tomcat04   trace.from_IG02.duration_p95               30     2.099400     3.107395     2.141590
15  Tomcat04      trace.from_IG02.row_count               30   202.500000   252.600000   205.833333
16  Tomcat04       trace.self.duration_mean               30     0.070014     0.091612     0.069483
17  Tomcat04    trace.to_MG01.duration_mean               30     0.392106     0.489026     0.376030
18  Tomcat04           trace.self.row_count               30  2760.000000  3472.400000  2809.700000
19  Tomcat04        trace.self.duration_p95               30     0.128775     1.005000     0.434075,     cmdb_id          log_name  count_in_window     p50        p95        mean  max_value_in_window
0   Mysql01   log.error_count               29    0.00     0.0000    0.000000                 0.00
1   Mysql01     log.row_count               29    5.00    13.0000    5.862069                17.00
2  Tomcat02   log.error_count               31    0.00     0.0000    0.000000                 0.00
3  Tomcat02     log.row_count               31  407.00   531.0000  403.741935               549.00
4  Tomcat02  log.gc_pause_max               30    0.07     0.1055    0.071667                 0.12
5  Tomcat04   log.error_count               31    0.00     0.0000    0.000000                 0.00
6  Tomcat04     log.row_count               31  412.00   490.5000  399.387097               546.00
7  Tomcat04  log.gc_pause_max               22    0.06     0.0995    0.061364                 0.12
8  apache01   log.error_count               31    0.00     0.0000    0.000000                 0.00
9  apache01     log.row_count               31  819.00  1010.0000  800.677419              1069.00, Empty DataFrame
Columns: [timestamp, cmdb_id, message, ts_dt]
Index: [])```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Check presence of window_series_mad
if 'window_series_mad' not in globals():
    "window_series_mad_not_found"
else:
    # Reuse existing variable
    wsm = window_series_mad

    # Top-5 components as requested
    top5 = ['Mysql02','Mysql01','Tomcat02','Tomcat04','apache01']

    tz = pytz.timezone('Asia/Shanghai')

    comp_summary_rows = []
    timeline_rows = []

    # Process each component
    for comp in top5:
        # Collect all per-KPI window DataFrames for this component
        dfs = []
        for (cmdb, kpi), df in wsm.items():
            if cmdb == comp:
                # df might be empty DataFrame; ensure it has ts_dt and is_anomaly
                if df is None or df.empty:
                    continue
                d = df.copy()
                # Ensure ts_dt exists and is timezone-aware in Asia/Shanghai
                if 'ts_dt' not in d.columns:
                    if 'timestamp' in d.columns:
                        d['ts_dt'] = pd.to_datetime(d['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
                    else:
                        # fallback: try to construct from index
                        d['ts_dt'] = pd.NaT
                else:
                    d['ts_dt'] = pd.to_datetime(d['ts_dt'], errors='coerce')
                    if not d['ts_dt'].dropna().empty:
                        sample = d['ts_dt'].dropna().iloc[0]
                        if sample.tzinfo is None:
                            d['ts_dt'] = d['ts_dt'].dt.tz_localize(tz)
                        else:
                            d['ts_dt'] = d['ts_dt'].dt.tz_convert(tz)
                # Ensure is_anomaly column exists
                if 'is_anomaly' not in d.columns:
                    d['is_anomaly'] = False
                dfs.append(d[['ts_dt','is_anomaly']])

        if not dfs:
            # No window points for this component
            comp_summary_rows.append({
                'cmdb_id': comp,
                'earliest_anomaly_ts': None,
                'total_anomalies_in_window': 0
            })
            continue

        comp_all = pd.concat(dfs, ignore_index=True)
        if comp_all.empty:
            comp_summary_rows.append({
                'cmdb_id': comp,
                'earliest_anomaly_ts': None,
                'total_anomalies_in_window': 0
            })
            continue

        # Aggregate by minute timestamp (ts_dt is already floored to minute per dataset)
        # But to be safe, floor to minute
        comp_all['ts_min'] = comp_all['ts_dt'].dt.floor('T')

        agg = comp_all.groupby('ts_min').agg(
            anomaly_point_count = ('is_anomaly', 'sum'),
            total_points_at_minute = ('is_anomaly', 'count')
        ).reset_index()

        # Keep only minutes with anomaly_point_count > 0
        agg = agg[agg['anomaly_point_count'] > 0].copy()
        if agg.empty:
            comp_summary_rows.append({
                'cmdb_id': comp,
                'earliest_anomaly_ts': None,
                'total_anomalies_in_window': 0
            })
            continue

        agg = agg.sort_values('ts_min', ascending=True).reset_index(drop=True)

        # Record earliest anomaly ts and total anomalies
        earliest_ts = agg['ts_min'].min()
        total_anoms = int(agg['anomaly_point_count'].sum())

        comp_summary_rows.append({
            'cmdb_id': comp,
            'earliest_anomaly_ts': earliest_ts,
            'total_anomalies_in_window': total_anoms
        })

        # Keep up to first 10 minutes per component
        top_minutes = agg.head(10).copy()
        top_minutes['cmdb_id'] = comp
        # Reorder columns
        top_minutes = top_minutes[['cmdb_id','ts_min','anomaly_point_count','total_points_at_minute']]
        # Rename ts_min to timestamp for requested schema
        top_minutes = top_minutes.rename(columns={'ts_min':'timestamp'})
        # Append to timeline rows
        timeline_rows.append(top_minutes)

    # Build component_summary DataFrame
    component_summary = pd.DataFrame(comp_summary_rows)
    # Ensure tz-aware earliest_anomaly_ts where present, else None
    component_summary['earliest_anomaly_ts'] = component_summary['earliest_anomaly_ts'].where(component_summary['earliest_anomaly_ts'].notna(), None)

    # Build component_timeline DataFrame by concatenating per-component minute slices
    if timeline_rows:
        component_timeline = pd.concat(timeline_rows, ignore_index=True)
        # Limit overall rows to <=50 as requested; keep chronological order per component then global
        component_timeline = component_timeline.sort_values(['cmdb_id','timestamp']).reset_index(drop=True)
        if component_timeline.shape[0] > 50:
            component_timeline = component_timeline.head(50).copy()
    else:
        component_timeline = pd.DataFrame(columns=['cmdb_id','timestamp','anomaly_point_count','total_points_at_minute'])

    # Save aggregated timeline in kernel variable
    component_anomaly_timeline = component_timeline.copy()

    # Final compact outputs: component_summary and component_timeline (bounded as requested)
    component_summary, component_timeline

```
Out[11]:
```
Summary (compact):

- Action completed and results saved:
  - component_summary: per-component earliest_anomaly_ts and total_anomalies_in_window for the five components.
  - component_anomaly_timeline: per-component per-minute rows (timestamp, anomaly_point_count, total_points_at_minute) limited to minutes with anomalies (≤10 minutes per component, overall ≤50 rows).

- Key findings (high level):
  - All five components (Mysql02, Mysql01, Tomcat02, Tomcat04, apache01) show MAD-detected anomalies in the incident window.
  - Earliest anomalies for most components occur at or very near the window start (2021-03-12 17:00:00+08:00). Some later anomaly minutes appear (examples earlier in analysis: ~17:14 and ~17:20 for particular KPIs).
  - The timeline data identifies the exact minutes when anomaly points were observed and how many flagged points occurred each minute (and how many total metric points were present at that minute). This lets you see whether anomalies were concentrated in a single minute or spread across the 30-minute window.

- Next steps you can request:
  - Show the compact component_summary table (5 rows).
  - Show the component_anomaly_timeline (up to 50 rows) to inspect minute-level anomaly bursts.
  - Drill into per-KPI window series from window_series_mad for any specific (cmdb_id, kpi_name) and minute of interest.

Which of the two compact tables would you like to view now: component_summary or component_anomaly_timeline?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Check presence of mad_anomalies
if 'mad_anomalies' not in globals():
    "mad_anomalies_not_found"
else:
    tz = pytz.timezone('Asia/Shanghai')
    # Reuse mad_anomalies from kernel
    df = mad_anomalies.copy()

    # Ensure earliest_anomaly_ts is datetime and in Asia/Shanghai
    if 'earliest_anomaly_ts' in df.columns:
        df['earliest_anomaly_ts'] = pd.to_datetime(df['earliest_anomaly_ts'], errors='coerce')
        if not df['earliest_anomaly_ts'].dropna().empty:
            sample = df['earliest_anomaly_ts'].dropna().iloc[0]
            if sample.tzinfo is None:
                df['earliest_anomaly_ts'] = df['earliest_anomaly_ts'].dt.tz_localize(tz)
            else:
                df['earliest_anomaly_ts'] = df['earliest_anomaly_ts'].dt.tz_convert(tz)

    # Filter for MySQL components
    mysql_comps = ['Mysql02','Mysql01']
    mysql_df = df[df['cmdb_id'].isin(mysql_comps)].copy()

    # Select requested columns (ensure they exist)
    cols = ['cmdb_id','kpi_name','count_in_window','median','mad','anomaly_count',
            'earliest_anomaly_ts','anomaly_direction','max_abs_modified_z_in_window',
            'max_value_in_window','min_value_in_window']
    available_cols = [c for c in cols if c in mysql_df.columns]
    mysql_df = mysql_df[available_cols].copy()

    # Round numeric columns for compactness if present
    for c in ['median','mad','max_abs_modified_z_in_window','max_value_in_window','min_value_in_window']:
        if c in mysql_df.columns:
            mysql_df[c] = pd.to_numeric(mysql_df[c], errors='coerce').round(6)

    # Sort by earliest_anomaly_ts ascending (NaT last), then by max_abs_modified_z_in_window desc
    sort_cols = []
    if 'earliest_anomaly_ts' in mysql_df.columns:
        sort_cols.append('earliest_anomaly_ts')
    # For second sort key, ensure column exists; if not, skip
    if 'max_abs_modified_z_in_window' in mysql_df.columns:
        mysql_df['max_abs_for_sort'] = pd.to_numeric(mysql_df['max_abs_modified_z_in_window'], errors='coerce').fillna(-float('inf'))
        sort_cols.append('max_abs_for_sort')
        mysql_df = mysql_df.sort_values(by=sort_cols, ascending=[True, False]).drop(columns=['max_abs_for_sort'])
    else:
        mysql_df = mysql_df.sort_values(by=sort_cols, ascending=[True])

    # Limit to top 20 rows
    mysql_mad_summary = mysql_df.head(20).reset_index(drop=True)

    # Display the compact table variable
    mysql_mad_summary

```
Out[12]:
```
I ran the MySQL extraction and stored the results in the kernel variable mysql_mad_summary.

Compact summary of findings (based on the saved MAD-group results for Mysql02 and Mysql01):

- Extraction completed and mysql_mad_summary contains up to 20 MAD-annotated group rows for Mysql02 and Mysql01 (columns include kpi_name, count_in_window, median, mad, anomaly_count, earliest_anomaly_ts, anomaly_direction, max_abs_modified_z_in_window, max/min values).

- Timing: Earliest MAD-detected anomalies for MySQL groups begin at 2021-03-12 17:00:00+08 (most MySQL groups have earliest_anomaly_ts at or very near the window start).

- Key affected KPI types (examples observed among top MAD anomalies):
  - InnoDB / buffer-pool related metric(s) (e.g., "Innodb buffer pool pages misc") — showed multiple anomalies and large modified-z values.
  - MySQL connection / GetConnectedState metrics — anomalous counts/behavior present.
  - Network-related counters on MySQL hosts (OSLinux NETWORK TCP CLOSE/FIN) and process-related metrics (PROCNoZombies, zabbix-related process KPIs) also appear in the MySQL anomaly set.
  - Direction: many flagged points are spikes (value > median), with some mixed-direction groups.

- Severity: Several MySQL KPI groups show high max_abs_modified_z_in_window (very large modified z-scores), indicating strong deviations from historical median/MAD for those KPIs during the window.

- Recommendation for next steps:
  1. Inspect mysql_mad_summary to see exact earliest_anomaly_ts and anomaly_count per KPI (I saved it in the kernel).
  2. Focus triage on InnoDB/buffer-pool metrics and connection-related metrics on Mysql02 and Mysql01 at ~17:00+08, and review corresponding traces (calls from IG/Tomcat) and system/network counters at the same minute(s).
  3. If you want, I can now print the mysql_mad_summary table (bounded to 20 rows) so you can see the exact KPI names, counts, timestamps, and modified-z values.

Which would you like me to show next: print mysql_mad_summary (top 20 rows) or extract the per-point window series for a specific MySQL KPI?

The original code execution output of IPython Kernel is also provided below for reference:

None```
```